# **ResNet50 — Baseline Final (Sintesis EXP03 + ResNet18 + ViT exp01)**

Notebook ini menyatukan praktik terbaik dari 3 sumber:
- **EXP03_Layer.ipynb** (ResNet50 lama) — starting point, tapi `OneCycleLR`-nya terbukti jadi root cause instabilitas fold (lihat diskusi sebelumnya: Fold 4 selalu menang karena satu-satunya yang tidak pernah early-stop sebelum siklus LR selesai)
- **cnn-resnet18.ipynb** — sudah pakai `timm`, Kaggle Secrets, `ReduceLROnPlateau`, struktur modular `train_one_fold()`
- **exp01-vit-base-16-amp.ipynb** — sudah pakai AMP, class weight ternormalisasi, pin_memory, cudnn.benchmark

**Yang diselaraskan lintas SEMUA arsitektur** (biar perbandingan CNN vs ViT nanti adil, tidak ada confound): FC head sederhana (`drop_rate`, bukan custom multi-layer), scheduler `ReduceLROnPlateau` dengan setting yang sama persis, kriteria checkpoint & early stopping berbasis `val_f1`, augmentasi "medium" yang identik.

---

## Update: EXP02 — Discriminative Learning Rate (isolasi tunggal dari baseline EXP01)

**Hipotesis**: LR seragam (1e-4) di semua layer unfreeze (`layer2, layer3, layer4, fc`) berpotensi merusak fitur pretrained di layer awal (`layer2`, lebih general/low-level) karena ikut diupdate sebesar layer akhir yang memang butuh banyak adaptasi ke domain skin disease. Ini didukung pola EXP01 lama (Layer4+FC saja, F1 0,6366) vs EXP03 lama (Layer2/3/4+FC, F1 0,5499) — makin banyak layer awal ikut di-unfreeze dengan LR seragam, makin turun performanya.

**Yang diubah dari baseline**: HANYA learning rate per-layer (lihat `DISCRIMINATIVE_LR` di Config). Semua yang lain — batch size, scheduler, patience, augmentasi, dropout, freeze pattern (layer mana yang unfreeze), seed — TETAP PERSIS SAMA dengan baseline. Ini supaya kalau hasilnya naik atau turun, penyebabnya jelas cuma dari discriminative LR ini (prinsip isolasi yang sama dipakai di roadmap ablation ViT).

Set `USE_DISCRIMINATIVE_LR = False` di Config untuk kembali ke perilaku baseline (LR flat) kalau perlu menjalankan ulang baseline dari notebook yang sama.

## 1. Import & Setup

In [1]:
# 1. Install & Import
import os, copy, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# PERBAIKAN (dari ViT exp01): AMP -- sebagian besar operasi forward jalan di
# float16 (lebih cepat & hemat VRAM di GPU RTX/Tensor Core), backward/update tetap
# presisi lewat GradScaler. Belum ada di kedua notebook ResNet sebelumnya.
from torch.cuda.amp import autocast, GradScaler

from torchvision import transforms, datasets
from PIL import Image
from tqdm import tqdm

import timm   # PERBAIKAN: torchvision.models -> timm, biar 1 API dipakai semua arsitektur (ResNet18/50, EfficientNet, ViT, dst)
import wandb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Ambil API key dari environment variable
wandb_api_key = os.getenv("WANDB_API_KEY")

os.environ["TORCH_HOME"] = "D:/cache/torch"
os.environ["HF_HOME"] = "D:/cache/huggingface"

os.environ["WANDB_DIR"] = "D:/cache/wandb"
os.environ["WANDB_CACHE_DIR"] = "D:/cache/wandb_cache"

os.environ["TEMP"] = "D:/cache/temp"
os.environ["TMP"] = "D:/cache/temp"

os.environ["CUDA_CACHE_PATH"] = "D:/cache/cuda"

print(os.getcwd())

# Login ke wandb
wandb.login(key=wandb_api_key)

# PERBAIKAN (dari ViT exp01 + ResNet18): seed eksplisit -- EXP01-03 ResNet50 lama
# cuma nge-seed StratifiedKFold, TIDAK nge-seed init bobot FC head / urutan shuffle.
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# PERBAIKAN (dari ViT exp01): cudnn.benchmark auto-tune algoritma konvolusi
# tercepat untuk ukuran input yang konsisten (semua di-resize ke 224x224).
torch.backends.cudnn.benchmark = True


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\UNIDA\_netrc.


d:\Devianest_SkripsiTest\Code_CNN


wandb: Currently logged in as: devianestnarendra to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Device: cuda


## 2. Config

In [2]:
TRAIN_DIR = r"D:\Devianest_SkripsiTest\train"
TEST_DIR  = r"D:\Devianest_SkripsiTest\test"

# PERBAIKAN (VSCode lokal): ganti "/kaggle/working" -> folder "outputs" relatif
# terhadap lokasi notebook ini. os.makedirs(..., exist_ok=True) otomatis bikin
# foldernya kalau belum ada, supaya tidak error "No such file or directory"
# saat pertama kali disimpan.
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ARCH_KEY  = "EXP01_ResNet50_Baseline"   # PERBAIKAN: nama baru -- eksperimen
                                                  # terpisah dari baseline EXP01, supaya checkpoint
                                                  # (.pth) dan run W&B tidak saling menimpa, dan hasil
                                                  # baseline tetap utuh sebagai pembanding.
TIMM_NAME = "resnet50"

IMG_SIZE     = 224
BATCH_SIZE   = 32          # tetap seperti EXP01-03 (ResNet50 lebih berat dari ResNet18, batch lebih kecil)
EPOCHS       = 50
N_FOLDS      = 5
DROPOUT      = 0.3         # dipertahankan dari EXP01-03 (nilai yang sudah teruji utk ResNet50)
LR           = 1e-4        # PERBAIKAN: dipertahankan nilai ResNet (BUKAN LR ViT 3e-5) -- CNN historically
                            # lebih toleran ke LR sedikit lebih tinggi dibanding attention layer ViT yang sensitif
WEIGHT_DECAY = 1e-4         # dipertahankan nilai konvensi CNN transfer learning (bukan WD ViT 0.01)
LABEL_SMOOTHING = 0.1
EARLY_STOP_PATIENCE = 7     # PERBAIKAN: naik dari 5 -> 7 (lebih toleran sebelum berhenti)

WANDB_PROJECT = "SkinDisease-CNN"   # disamakan dengan notebook ResNet18, biar 1 project W&B

# ResNet50: unfreeze layer2/3/4 + fc -- sama scope kapasitas dengan EXP03 (untuk
# tetap bisa dibandingkan), TAPI training regime-nya sudah diperbaiki (lihat bawah).
UNFREEZE_PATTERNS = ["layer2", "layer3", "layer4", "fc"]

# PERBAIKAN (dari ViT exp01 + ResNet18): scheduler ReduceLROnPlateau disamakan
# PERSIS dengan setting yang sudah dipakai di ViT & ResNet18 -- root cause
# instabilitas ResNet50 lama adalah OneCycleLR yang di-set untuk siklus 50 epoch
# penuh, tapi EarlyStopping hampir selalu memotong training di epoch 11-22
# (SEBELUM fase anneal selesai) -- lihat diskusi sebelumnya soal Fold 4 yang
# selalu menang karena satu-satunya fold yang tidak pernah early-stop.
SCHEDULER_FACTOR    = 0.1
SCHEDULER_PATIENCE  = 2
SCHEDULER_THRESHOLD = 1e-4
SCHEDULER_MIN_LR    = 1e-7


# ── EKSPERIMEN: Discriminative Learning Rate (isolasi tunggal) ──────────────
# HANYA parameter ini yang beda dari baseline EXP01. Semua config lain di atas
# (BATCH_SIZE, EPOCHS, DROPOUT, WEIGHT_DECAY, LABEL_SMOOTHING, SCHEDULER_*,
# UNFREEZE_PATTERNS, EARLY_STOP_PATIENCE) TIDAK diubah -- supaya efek yang
# terukur murni dari LR per-layer, bukan confound faktor lain.
#
# Skema: LR kecil di layer paling dekat input (layer2, fitur general/low-level
# dari ImageNet, tidak butuh banyak adaptasi), makin besar ke layer akhir yang
# paling perlu adaptasi ke domain skin disease. layer4 & fc sengaja disamakan
# dengan LR baseline (1e-4) supaya bisa dibaca sebagai "titik referensi" yang
# tidak berubah -- perbandingan ke baseline jadi lebih bersih.
USE_DISCRIMINATIVE_LR = True   # set False untuk kembali ke LR flat (perilaku baseline EXP01)

DISCRIMINATIVE_LR = {
    "layer2": 1e-5,   # dekat input, fitur general -- update paling halus
    "layer3": 5e-5,   # transisi
    "layer4": 1e-4,   # dekat output, paling perlu adaptasi -- LR = baseline
    "fc":     1e-4,   # head baru (random init) -- disamakan LR baseline dulu,
                      # supaya isolasi cuma di layer backbone, bukan di head
}


## 3. Dataset & Augmentasi

In [3]:
# PERBAIKAN: augmentasi disamakan PERSIS dengan versi terbaru ViT exp01 (medium,
# termasuk RandomResizedCrop) -- bukan versi ResNet18 yang belum pakai
# RandomResizedCrop. Ini penting supaya CNN dan ViT dibandingkan dengan
# preprocessing yang identik (bukan confound tambahan).
def get_transforms(img_size):
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return train_tf, eval_tf


classes = sorted(os.listdir(TRAIN_DIR))
class_to_idx = {c: i for i, c in enumerate(classes)}
num_classes = len(classes)

filepaths, labels = [], []
for label in classes:
    class_path = os.path.join(TRAIN_DIR, label)
    for img in os.listdir(class_path):
        filepaths.append(os.path.join(class_path, img))
        labels.append(class_to_idx[label])

print("Total Images :", len(filepaths))
print("Classes      :", num_classes)


class SkinDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


Total Images : 15557
Classes      : 23


## 4. Early Stopping (val_f1) + K-Fold

In [4]:
class EarlyStopping:
    # Kriteria val_f1 (bukan val_loss) -- konsisten dengan ViT exp01 & ResNet18:
    # lebih robust untuk data imbalanced (23 kelas DermNet) dibanding val_loss.
    def __init__(self, patience=5):
        self.patience = patience
        self.best_f1 = -np.inf
        self.counter = 0

    def step(self, val_f1):
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


## 5. Model Builder + Freeze Strategy

In [5]:
def build_model(num_classes, dropout=DROPOUT):
    # PERBAIKAN: head sederhana bawaan timm (Linear + drop_rate), BUKAN custom
    # Linear->BN->ReLU->Dropout->Linear seperti EXP01-03 lama. Diselaraskan dengan
    # ViT exp01 & ResNet18 -- supaya kapasitas head TIDAK jadi confound tambahan
    # saat membandingkan arsitektur (kalau satu arsitektur dikasih head lebih besar
    # dari yang lain, selisih performa bisa jadi cuma soal head, bukan backbone).
    model = timm.create_model(
        TIMM_NAME, pretrained=True, num_classes=num_classes, drop_rate=dropout
    )
    return model


def apply_freeze_strategy(model, patterns=UNFREEZE_PATTERNS):
    for p in model.parameters():
        p.requires_grad = False
    for name, p in model.named_parameters():
        if any(pat in name for pat in patterns):
            p.requires_grad = True

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"  [{ARCH_KEY}] Trainable params: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.1f}%)")
    return model


def build_param_groups(model, lr_map=DISCRIMINATIVE_LR, base_lr=LR):
    # PERBAIKAN (eksperimen discriminative LR): pisahkan parameter trainable
    # jadi beberapa param_group dengan LR beda per pattern nama layer, alih-alih
    # 1 LR flat untuk semua (perilaku baseline EXP01). ReduceLROnPlateau tetap
    # jalan normal di sini -- ia mengalikan factor yang sama ke SEMUA group saat
    # plateau, jadi rasio antar-layer LR tetap terjaga sepanjang training.
    groups = []
    assigned_ids = set()
    for pattern, lr in lr_map.items():
        params = [p for n, p in model.named_parameters() if pattern in n and p.requires_grad]
        if params:
            groups.append({"params": params, "lr": lr, "name": pattern})
            assigned_ids.update(id(p) for p in params)

    # Fallback: parameter trainable yang tidak match pattern manapun di lr_map
    # (jaga-jaga kalau UNFREEZE_PATTERNS berubah tapi lr_map belum diupdate)
    remaining = [p for p in model.parameters() if p.requires_grad and id(p) not in assigned_ids]
    if remaining:
        groups.append({"params": remaining, "lr": base_lr, "name": "other"})

    n_groups_param = sum(len(g["params"]) for g in groups)
    n_trainable_total = sum(p.numel() for g in groups for p in g["params"])
    print(f"  [{ARCH_KEY}] Param groups: " +
          ", ".join(f"{g['name']}(lr={g['lr']:.1e})" for g in groups))
    return groups


## 6. Train 1 Fold

In [6]:
def train_one_fold(fold, train_idx, val_idx, run):
    train_tf, eval_tf = get_transforms(IMG_SIZE)

    train_files  = [filepaths[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    val_files    = [filepaths[i] for i in val_idx]
    val_labels   = [labels[i] for i in val_idx]

    # PERBAIKAN (dari ViT exp01): pin_memory=True -- percepat transfer CPU->GPU.
    # num_workers=2 dari ResNet18 (lebih cepat load data daripada 0 di EXP03 lama).
    train_loader = DataLoader(
        SkinDataset(train_files, train_labels, train_tf),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        SkinDataset(val_files, val_labels, eval_tf),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True
    )

    # PERBAIKAN: reseed per fold -- inisialisasi FC head & urutan shuffle
    # reproducible, tidak tergantung urutan eksekusi fold sebelumnya.
    random.seed(SEED + fold); np.random.seed(SEED + fold)
    torch.manual_seed(SEED + fold); torch.cuda.manual_seed_all(SEED + fold)

    model = build_model(num_classes)
    model = apply_freeze_strategy(model)
    model = model.to(device)

    # PERBAIKAN (dari ViT exp01): class_weights dinormalisasi supaya rata-rata = 1.
    # EXP01-03 & ResNet18 sebelumnya cuma 1./bincount TANPA normalisasi --
    # magnitude weight antar kelas timpang, jadi salah satu sumber loss yang
    # "melompat" tergantung komposisi kelas tiap batch.
    class_counts  = np.bincount(train_labels, minlength=num_classes)
    class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_weights / class_weights.sum() * num_classes

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device), label_smoothing=LABEL_SMOOTHING
    )
    # PERBAIKAN: discriminative LR (isolasi tunggal dari baseline) -- kalau
    # USE_DISCRIMINATIVE_LR=False, perilaku persis sama seperti EXP01 (LR flat).
    if USE_DISCRIMINATIVE_LR:
        param_groups = build_param_groups(model, DISCRIMINATIVE_LR, base_lr=LR)
        optimizer = optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
    else:
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR, weight_decay=WEIGHT_DECAY
        )
    # PERBAIKAN: scheduler disamakan persis dengan ViT exp01 & ResNet18 -- root
    # cause instabilitas ResNet50 lama (OneCycleLR + EarlyStopping yang memotong
    # sebelum siklus selesai) sudah tidak ada lagi di sini.
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE,
        threshold=SCHEDULER_THRESHOLD, min_lr=SCHEDULER_MIN_LR
    )

    # PERBAIKAN (dari ViT exp01): AMP -- autocast di forward pass, GradScaler
    # untuk backward/update supaya gradient float16 tidak underflow.
    scaler = GradScaler()

    early_stopping = EarlyStopping(patience=EARLY_STOP_PATIENCE)
    best_val_f1 = -np.inf
    # PERBAIKAN: tracking accuracy/precision/recall di titik checkpoint terbaik juga
    # (sebelumnya cuma f1) -- supaya format output/summary sama seperti ViT exp01,
    # yang melaporkan keempat metrik (bukan cuma F1) di rekap akhir.
    best_val_acc = -np.inf
    best_val_precision = -np.inf
    best_val_recall = -np.inf
    best_val_loss = np.inf
    best_train_loss = np.inf
    best_model_path = None
    train_losses, val_losses = [], []

    for epoch in range(EPOCHS):
        # PERBAIKAN: tampilkan LR tiap param_group (bukan cuma group pertama) --
        # penting karena sekarang tiap layer bisa punya LR beda (discriminative LR).
        lr_display = ", ".join(
            f"{g.get('name', i)}={g['lr']:.1e}" for i, g in enumerate(optimizer.param_groups)
        )
        print(f"\n[{ARCH_KEY} | fold {fold+1}] Epoch {epoch+1}/{EPOCHS} (LR: {lr_display})")

        # TRAIN
        model.train()
        train_loss = 0
        for imgs, tgts in tqdm(train_loader, desc="Train"):
            imgs, tgts = imgs.to(device), tgts.to(device)
            optimizer.zero_grad()
            with autocast():
                loss = criterion(model(imgs), tgts)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        # VALIDATION
        model.eval()
        val_loss = 0
        preds, trues = [], []
        with torch.no_grad():
            for imgs, tgts in tqdm(val_loader, desc="Val"):
                imgs, tgts = imgs.to(device), tgts.to(device)
                with autocast():
                    outputs = model(imgs)
                    v_loss  = criterion(outputs, tgts)
                val_loss += v_loss.item()
                preds.extend(outputs.argmax(1).cpu().numpy())
                trues.extend(tgts.cpu().numpy())

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss   = val_loss / len(val_loader)

        acc       = accuracy_score(trues, preds)
        precision = precision_score(trues, preds, average="weighted", zero_division=0)
        recall    = recall_score(trues, preds, average="weighted", zero_division=0)
        f1        = f1_score(trues, preds, average="weighted", zero_division=0)

        scheduler.step(f1)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f"Train Loss : {avg_train_loss:.4f} | Val Loss  : {avg_val_loss:.4f}")
        print(f"Accuracy   : {acc:.4f}  | Precision : {precision:.4f}")
        print(f"Recall     : {recall:.4f}  | F1 Score  : {f1:.4f}")

        run.log({
            "epoch": epoch + 1,
            f"fold_{fold+1}/train_loss": avg_train_loss,
            f"fold_{fold+1}/val_loss": avg_val_loss,
            f"fold_{fold+1}/accuracy": acc,
            f"fold_{fold+1}/precision": precision,
            f"fold_{fold+1}/recall": recall,
            f"fold_{fold+1}/f1_score": f1,
            # PERBAIKAN: log LR tiap param_group secara terpisah (mis. fold_1/lr_layer2,
            # fold_1/lr_layer4, dst) supaya bisa dipantau di W&B apakah ReduceLROnPlateau
            # menurunkan semua group secara proporsional seperti yang diharapkan.
            **{f"fold_{fold+1}/lr_{g.get('name', i)}": g["lr"]
               for i, g in enumerate(optimizer.param_groups)},
        })

        # SAVE BEST MODEL -- kriteria val_f1 tertinggi (bukan val_loss terendah)
        if f1 > best_val_f1:
            best_val_f1 = f1
            best_val_acc = acc
            best_val_precision = precision
            best_val_recall = recall
            best_val_loss = avg_val_loss
            best_train_loss = avg_train_loss

            save_path = f"{OUTPUT_DIR}/{ARCH_KEY}_fold{fold+1}.pth"
            torch.save({
                "model_state_dict": model.state_dict(),
                "val_loss": avg_val_loss,
                "f1": f1,
                "fold": fold + 1,
                "arch": ARCH_KEY,
            }, save_path)
            best_model_path = save_path
            print(f"  ✓ Model saved → {save_path} (F1: {f1:.4f})")

        if early_stopping.step(f1):
            print("Early Stopping Triggered")
            break

    # ── PLOT LOSS CURVE PER FOLD ──────────────────────────────────────────────
    epochs_ran = range(1, len(train_losses) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs_ran, train_losses, label="Train Loss", marker="o", markersize=3)
    ax.plot(epochs_ran, val_losses, label="Val Loss", marker="o", markersize=3)
    ax.set_title(f"{ARCH_KEY} — Fold {fold+1} Loss Curve")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    curve_path = f"{OUTPUT_DIR}/{ARCH_KEY}_Fold_{fold+1}_Loss_Curve.png"
    fig.savefig(curve_path, dpi=150, bbox_inches="tight")
    run.log({f"Loss_Curve/Fold_{fold+1}": wandb.Image(curve_path)})
    plt.close(fig)

    return {
        "arch": ARCH_KEY,
        "fold": fold + 1,
        "train_loss": best_train_loss,
        "val_loss": best_val_loss,
        # PERBAIKAN: sertakan accuracy/precision/recall (bukan cuma f1), supaya
        # results_df punya kolom yang sama seperti fold_accuracies/fold_precision/
        # fold_recall/fold_f1 di ViT exp01.
        "accuracy": best_val_acc,
        "precision": best_val_precision,
        "recall": best_val_recall,
        "f1": best_val_f1,
        "model_path": best_model_path,
    }


## 7. MAIN LOOP — 5 Fold (ResNet50)

Kalau waktu habis di tengah jalan: checkpoint tiap fold udah ke-save duluan (di `all_results`), aman buat dilanjut manual per-fold.

In [7]:
all_results = []

run = wandb.init(
    project="SkinDisease-CNN",
    entity="devianestnarendra_Team",
    name=f"{ARCH_KEY}",
    reinit=True,
    config={
        "architecture": ARCH_KEY,
        "n_folds": N_FOLDS,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "optimizer": "AdamW",
        "scheduler": f"ReduceLROnPlateau(mode=max, factor={SCHEDULER_FACTOR}, patience={SCHEDULER_PATIENCE})",
        "lr": LR,
        "use_discriminative_lr": USE_DISCRIMINATIVE_LR,
        "discriminative_lr": DISCRIMINATIVE_LR if USE_DISCRIMINATIVE_LR else None,
        "dropout": DROPOUT,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "unfreeze_patterns": UNFREEZE_PATTERNS,
        "checkpoint_criteria": "best_val_f1",
        "amp": True,
        "seed": SEED,
    }
)

for fold, (train_idx, val_idx) in enumerate(skf.split(filepaths, labels)):
    result = train_one_fold(fold, train_idx, val_idx, run)
    all_results.append(result)

run.finish()

results_df = pd.DataFrame(all_results)
results_df


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


  [EXP01_ResNet50_Baseline] Trainable params: 23,329,815 / 23,555,159 (99.0%)


C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15020\86009615.py:60: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


  [EXP01_ResNet50_Baseline] Param groups: layer2(lr=1.0e-05), layer3(lr=5.0e-05), layer4(lr=1.0e-04), fc(lr=1.0e-04)

[EXP01_ResNet50_Baseline | fold 1] Epoch 1/50 (LR: layer2=1.0e-05, layer3=5.0e-05, layer4=1.0e-04, fc=1.0e-04)


Train:   0%|          | 0/389 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15020\86009615.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/98 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15020\86009615.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val: 100%|██████████| 98/98 [00:38<00:00,  2.54it/s]


...

Train Loss : 1.0878 | Val Loss  : 2.3607
Accuracy   : 0.5436  | Precision : 0.5624
Recall     : 0.5436  | F1 Score  : 0.5456

[EXP01_ResNet50_Baseline | fold 5] Epoch 50/50 (LR: layer2=1.0e-07, layer3=5.0e-07, layer4=1.0e-06, fc=1.0e-06)


Train:   0%|          | 0/389 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15020\86009615.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/98 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15020\86009615.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val: 100%|██████████| 98/98 [00:26<00:00,  3.73it/s]


Train Loss : 1.0952 | Val Loss  : 2.3614
Accuracy   : 0.5468  | Precision : 0.5678
Recall     : 0.5468  | F1 Score  : 0.5484


epoch,▁▃▄▅▅▆▇▇▂▃▅▅▆▆▆▇█▁▁▂▃▃▄▄▅▆▆▆▇▇▁▅▅▆▇▄▅▆▆█
fold_1/accuracy,▁▂▃▃▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████████████
fold_1/f1_score,▁▂▃▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇██████████████
fold_1/lr_fc,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
fold_1/lr_layer2,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
fold_1/lr_layer3,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
fold_1/lr_layer4,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
fold_1/precision,▁▃▃▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇█▇█▇██████████
fold_1/recall,▁▂▃▃▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█████████████
fold_1/train_loss,█▇▇▆▆▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
+41,...


,arch,fold,train_loss,val_loss,accuracy,precision,recall,f1,model_path
0,EXP01_ResNet50_Baseline,1,1.001581,2.312122,0.566517,0.580355,0.566517,0.568360,outputs/EXP01_ResNet50_Baseline_fold1.pth
1,EXP01_ResNet50_Baseline,2,1.017118,2.339622,0.564589,0.579879,0.564589,0.566780,outputs/EXP01_ResNet50_Baseline_fold2.pth
2,EXP01_ResNet50_Baseline,3,1.065179,2.314024,0.564449,0.579758,0.564449,0.567450,outputs/EXP01_ResNet50_Baseline_fold3.pth
3,EXP01_ResNet50_Baseline,4,1.143813,2.338684,0.556091,0.573387,0.556091,0.558099,outputs/EXP01_ResNet50_Baseline_fold4.pth
4,EXP01_ResNet50_Baseline,5,1.092523,2.355745,0.555770,0.574810,0.555770,0.558244,outputs/EXP01_ResNet50_Baseline_fold5.pth


## 8. Rekap 5-Fold

In [8]:
# PERBAIKAN: format summary disamakan persis dengan ViT exp01 -- laporkan
# Mean ± Std untuk keempat metrik (Accuracy, Precision, Recall, F1), bukan
# cuma F1 saja.
print("\n" + "="*50)
print(f"  FINAL RESULT — ALL FOLDS ({ARCH_KEY})")
print("="*50)
print(f"Mean Accuracy  : {results_df['accuracy'].mean():.4f} ± {results_df['accuracy'].std():.4f}")
print(f"Mean Precision : {results_df['precision'].mean():.4f} ± {results_df['precision'].std():.4f}")
print(f"Mean Recall    : {results_df['recall'].mean():.4f} ± {results_df['recall'].std():.4f}")
print(f"Mean F1 Score  : {results_df['f1'].mean():.4f} ± {results_df['f1'].std():.4f}")

# ── WANDB LOG SUMMARY (format sama seperti ViT exp01) ─────────────────────────
try:
    run.log({
        "summary/mean_accuracy"  : results_df["accuracy"].mean(),
        "summary/mean_precision" : results_df["precision"].mean(),
        "summary/mean_recall"    : results_df["recall"].mean(),
        "summary/mean_f1"        : results_df["f1"].mean(),
        "summary/std_accuracy"   : results_df["accuracy"].std(),
        "summary/std_precision"  : results_df["precision"].std(),
        "summary/std_recall"     : results_df["recall"].std(),
        "summary/std_f1"         : results_df["f1"].std(),
    })
except Exception as e:
    print(f"  ⚠ W&B log summary gagal (dilewati): {e}")

# ── GRAFIK: 4 metrik per fold (bukan cuma F1) ──────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
metric_cols = ["accuracy", "precision", "recall", "f1"]
metric_titles = ["Accuracy", "Precision", "Recall", "F1-Score"]

for ax, col, title in zip(axes, metric_cols, metric_titles):
    ax.bar(results_df["fold"].astype(str), results_df[col], color="#1f3a5f")
    ax.axhline(results_df[col].mean(), color="red", linestyle="--", label=f"Mean = {results_df[col].mean():.3f}")
    ax.set_xlabel("Fold"); ax.set_ylabel(f"Val {title}")
    ax.set_title(f"{ARCH_KEY} — {title} per Fold")
    ax.set_ylim(0, 1); ax.legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/{ARCH_KEY}_fold_metrics_chart.png", dpi=200)
plt.show()

results_df.to_csv(f"{OUTPUT_DIR}/{ARCH_KEY}_all_folds.csv", index=False)



  FINAL RESULT — ALL FOLDS (EXP01_ResNet50_Baseline)
Mean Accuracy  : 0.5615 ± 0.0051
Mean Precision : 0.5776 ± 0.0033
Mean Recall    : 0.5615 ± 0.0051
Mean F1 Score  : 0.5638 ± 0.0052
  ⚠ W&B log summary gagal (dilewati): Run (2vkzddtz) is finished. The call to `log` will be ignored. Please make sure that you are using an active run.


C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15020\1248490208.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Test Evaluation (Fold Terbaik)

In [9]:
best_fold_result = max(all_results, key=lambda r: r["f1"])
best_overall_path = best_fold_result["model_path"]
print(f"Fold terbaik    : {best_fold_result['fold']}")
print(f"Checkpoint      : {best_overall_path}")
print(f"Val F1 terbaik  : {best_fold_result['f1']:.4f}")

_, eval_tf = get_transforms(IMG_SIZE)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_tf)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = build_model(num_classes)
model = model.to(device)
checkpoint = torch.load(best_overall_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for imgs, tgts in tqdm(test_loader, desc="Test"):
        imgs = imgs.to(device)
        with autocast():
            out = model(imgs)
        y_true.extend(tgts.numpy())
        y_pred.extend(out.argmax(1).cpu().numpy())

acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(15, 15))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes).plot(cmap="Blues", ax=ax, xticks_rotation=90)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/{ARCH_KEY}_Test_Confusion_Matrix.png", dpi=150, bbox_inches="tight")
plt.show()

test_summary_df = pd.DataFrame([{
    "arch": ARCH_KEY, "Accuracy": acc, "Precision": precision, "Recall": recall, "F1": f1
}])
test_summary_df.to_csv(f"{OUTPUT_DIR}/{ARCH_KEY}_Test_Summary.csv", index=False)
print(f"\n✓ Test summary disimpan -> {OUTPUT_DIR}/{ARCH_KEY}_Test_Summary.csv")


Fold terbaik    : 1
Checkpoint      : outputs/EXP01_ResNet50_Baseline_fold1.pth
Val F1 terbaik  : 0.5684


Test:   0%|          | 0/126 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15020\3000596476.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Test: 100%|██████████| 126/126 [00:43<00:00,  2.90it/s]


Accuracy  : 0.5752
Precision : 0.5893
Recall    : 0.5752
F1-Score  : 0.5757

                                                                    precision    recall  f1-score   support

                                           Acne and Rosacea Photos       0.78      0.88      0.83       312
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions       0.73      0.51      0.60       288
                                          Atopic Dermatitis Photos       0.51      0.62      0.56       123
                                            Bullous Disease Photos       0.59      0.46      0.52       113
                Cellulitis Impetigo and other Bacterial Infections       0.34      0.41      0.38        73
                                                     Eczema Photos       0.59      0.60      0.59       309
                                      Exanthems and Drug Eruptions       0.45      0.57      0.50       101
                 Hair Loss Photos Alopecia and other Hair 

C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15020\3000596476.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Catatan

- Checkpoint format: `{model_state_dict, val_loss, f1, fold, arch}` -- kompatibel dengan notebook ResNet18 dan rencana perbandingan 5 arsitektur.
- `results_df` punya kolom yang sama (`arch, fold, train_loss, val_loss, f1, model_path`) -- tinggal `pd.concat()` dengan hasil arsitektur lain saat rekap akhir.
- **Rekomendasi lanjutan**: notebook ResNet18 kamu saat ini augmentasinya belum pakai `RandomResizedCrop` seperti di sini -- worth diselaraskan juga supaya semua arsitektur (termasuk ViT) benar-benar dilatih dengan preprocessing identik.
- Scope unfreeze (`layer2/3/4 + fc`) dipertahankan sama dengan EXP03 lama supaya masih bisa dibandingkan sebagai referensi "apakah masalahnya training regime atau kapasitas", sesuai hipotesis yang kita uji sebelumnya.

### Cara membaca hasil EXP02 vs EXP01 (baseline)

- Bandingkan `results_df` / summary W&B EXP02 dengan EXP01 secara langsung -- semua kolom formatnya sama.
- Kalau Mean F1 EXP02 > EXP01 dan deviasi antar fold tetap rendah (~sama atau lebih rendah dari 0,0278): discriminative LR terbukti membantu, lanjut eksplor rasio LR yang lain atau kombinasikan dengan perubahan berikutnya.
- Kalau Mean F1 EXP02 ≈ EXP01 atau malah turun: itu tetap temuan valid ("discriminative LR tidak signifikan pada scope unfreeze ini") -- jangan buru-buru ubah beberapa variabel sekaligus, cek dulu apakah rasio LR (1e-5/5e-5/1e-4/1e-4) kurang agresif, sebelum coba variabel lain.
- Jangan jalankan EXP02 dengan mengubah `BATCH_SIZE`/`SCHEDULER_PATIENCE`/dll di saat yang sama -- kalau mau coba itu, itu jadi EXP03 tersendiri, supaya penyebab naik/turunnya performa tetap bisa dilacak ke satu variabel.